# Step 5: Identify species

![Step 5 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/05-step.png)

## What you're building today

Yesterday we answered: *is there a bird?*
Today we answer: *what kind?*

We'll take the bird crop from the detection box, send it to a species classifier, and get a name back. Like "American Robin, 92% confident."

By the end, every bird frame will have a name on it.

## Step 5.1 — Detect vs identify, side by side

- **Detection** = finding WHERE the bird is (a box around it)
- **Identification** = figuring out WHICH bird it is (a name + confidence)

They're separate models trained for separate jobs. We chain them: first detect, then identify the crop.

For identification we'll use a HuggingFace model trained specifically on birds. Way better than asking YOLO "is this an American Robin?" — YOLO only knows it's a bird, not what kind.

In [ ]:
!pip install -q transformers torch torchvision Pillow
print("OK")

## Step 5.2 — Load the species classifier

We're using `dennisjooo/Birds-Classifier`, a small but accurate model hosted on HuggingFace. First run downloads it (~80MB), then it's cached.

It knows ~500 common North American and European bird species.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="image-classification",
    model="dennisjooo/Birds-Classifier",
)
print(f"Loaded classifier. Knows {len(classifier.model.config.id2label)} species.")

## Step 5.3 — Detect first, then crop the bird

We chain the two: YOLO finds the bird, then we slice that part of the image out and feed the slice to the species classifier.

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display
import requests
from pathlib import Path

RUNNING_IN_COLAB = "COLAB_GPU" in os.environ
if RUNNING_IN_COLAB:
    SOURCE = "https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/data/samples/sample-bird.jpg"
else:
    SOURCE = "http://192.168.1.42:8080/photo.jpg"

frame_path = Path("data/snapshots/id-input.jpg")
frame_path.write_bytes(requests.get(SOURCE, timeout=10).content)

detector = YOLO("yolov8n.pt")
results = detector(frame_path, verbose=False)

BIRD_CLASS_ID = 14
bird_boxes = [b for b in results[0].boxes
              if int(b.cls[0]) == BIRD_CLASS_ID and float(b.conf[0]) > 0.4]

if not bird_boxes:
    print("no bird — nothing to identify")
else:
    img = Image.open(frame_path)
    box = bird_boxes[0]
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    crop = img.crop((x1, y1, x2, y2))
    crop.save("data/snapshots/id-crop.jpg")
    print(f"Cropped bird: {crop.size}")
    display(crop)

## Step 5.4 — Identify the species

Hand the crop to the classifier. It returns its top guesses with confidence scores.

In [ ]:
predictions = classifier(crop)

# Print the top 3 guesses
for i, pred in enumerate(predictions[:3]):
    print(f"  {i+1}. {pred['label']:30s}  ({pred['score']:.2f})")

best = predictions[0]
print(f"\nspecies: {best['label']} ({best['score']:.2f})")

## Done?

If you saw something like:

```
species: American Robin (0.92)
```

...then the brain can name names. 🎉

If the species is wrong, don't worry — the classifier is ~85% accurate on common birds. As long as the right family is in the top 3, you've succeeded.

## What's next

Issue **#6: Persist sightings**. Right now every frame's species lives in the notebook, then disappears when you close it. Next we save them to a database so you can look back at what flew by. 🐦